# Face Detection System for a Digital Camera

<span style="color:red">**NB: All the diretories to the dataset folders and images refers to local paths**</span>

**Project Context**

**ProCam S.p.A.** is preparing to launch a new compact digital camera designed to be affordable and trendy for young photography enthusiasts. The product's primary goal is to simplify the shooting experience, specifically for solo or group selfies.

**The Challenge**

You have been hired as a **Data Scientist** to develop a **face detection system**. This system will assist technicians in automatically optimizing camera settings during selfie captures. Your task is to create a pipeline that identifies faces within images and returns the coordinates of the **bounding boxes** where those faces are located. If no faces are detected, the pipeline must return an empty list.

This is a **Computer Vision** problem—specifically, **Face Detection**.

**Project Requirements**

**Objective:** Build a face detection system using **Scikit-learn**. The pipeline must be capable of:

1. Receiving an input image.
2. Returning a list of bounding box coordinates for all detected faces.
3. Returning an empty list if no faces are present in the image.

**Constraints:**

* **Dataset:** No dataset is provided. You must find a suitable dataset online or, if no alternatives are available, construct one yourself.
* **Pre-trained Models:** The use of pre-trained models is **strictly prohibited**. The face detection model must be trained from scratch using Scikit-learn.
* **Computing Resources:** You will work on a system with limited computational capacity. The model must be optimized for low resource consumption.
* **Documentation:** The solution must be thoroughly documented. Every decision made (choice of algorithms, preprocessing, optimization techniques) must be explained. Furthermore, all external resources used (academic papers, blog posts, GitHub code) must be cited.
* **Literature Review:** Since detailed implementation instructions are not provided, it is essential to conduct an in-depth **literature review** to identify the best solutions. You will need to analyze existing approaches and adapt them to the project's constraints.

## Import

In [ ]:
import os
import io
import random
import zipfile
from tqdm import tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

# scikit-learn
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, accuracy_score
from sklearn.decomposition import PCA


from skimage.feature import hog
from skimage.color import rgb2gray
from skimage.transform import rescale

import cv2
import joblib
import shutil
import kagglehub

# Research

## Dataset

**Positive samples**

From [arxiv](https://arxiv.org/html/2511.17393v1#S2) I found some face datasets good for the positive samples:
- [LFW (Labeled Faces in the Wild)](https://www.kaggle.com/datasets/jessicali9530/lfw-dataset): 13,233 face images of real life condition (scikit-learn)
- [RFW (Racial Faces in the Wild)](http://whdeng.cn/RFW/testing.html): is a testing database for studying racial bias (Caucasian, Asian, Indian and African) in face recognition
- [AgeDB](https://complexity.cecs.ucf.edu/agedb/): contains 12,240 face images of celebrities, politicians and scientists in different ages and poses
- [CFPW (Celebrities in Frontal-Profile in the Wild)](https://www.kaggle.com/datasets/chinafax/cfpw-dataset): contains 7000 images of frontal and profile of celebrities
- [ROF (Real World Occulted Faces)](https://github.com/ekremerakin/RealWorldOccludedFaces): contains 3195 neutral images, 1686 sunglasses images and 678 masked images of human faces

- [WIDER FACE](http://shuoyang1213.me/WIDERFACE/): the gold standard dataset for face detection algorithms. This dataset collects 32,203 images and label 393,703 faces (each one with its bounding box) with a high degree of variability in scale, pose and occlusion. This could be perfect for this use case but the elaboration of the dataset requires much more work to extract the faces in the bounding boxes and will be longer. Maybe this could be considered in second hand.

- [FairFace](https://ar5iv.labs.arxiv.org/html/1908.04913): a dataset contains 108,501 facial images with 7 race groups defined: White, Black, Indian, East Asian, Southeast Asian, Middle East, and Latino. Like RFW the goal of this dataset is to overcame the racial bias in the computer vision models. The dataset is also balanced on age and gender.

- [Hybrid Face Detection Dataset](https://data.mendeley.com/datasets/s57wnx78vh/2): Occluded and Low-Light Facial Images is a comprehensive and diverse collection designed to support the development and evaluation of robust face detection algorithms under real-world challenges. This dataset contains a total of 12,000 images, primarily featuring human faces captured under a variety of difficult conditions.

**Negative Samples**

For the negative images the dataset mustn't have face people so we are going to research natural landscape images or cities images that can have some patterns (lines, curves) that allow the model to distinguish the landascape from a face.

Online there is a very large number of datasets including:
- [SUN Database: Scene Categorization Benchmark](https://3dvision.princeton.edu/projects/2010/SUN/) that is a database that contains 899 categories and 130,519 images about scenes and not human faces. (*PS: Unfortunately Princeton requires authentication to use those data so we need to move forward.*)
- [WIDER FACE](http://shuoyang1213.me/WIDERFACE/): an other choice could be download from WIDER FACE the patches out of the bounding boxes indicated. Despite this is a good resouce the work to extract those patch is very high.
- [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification): Kaggle dataset. This Data contains around 25k images of size 150x150 distributed under 6 categories: buildings, forest, glacier, mountain, sea, street. and is well designed to not have faces.

### Strategy

The strategy for the positives is building up a good dataset to train our model is to get 15000 images (with a balance distribution of the races) from *FairFace* because:
- is already well balanced on race, age, gender
- easier to process than the WIDER FACE dataset

The strategy for negatives is selecting 5000 images from Intel Image Classification dataset and then generate 30000 of patches 64x64 that is the size of our training images for the face recognition.

After a first train and tuning with the dataset of 45k images will be performed the **Hard Negative Mining strategy**. It means that the detector will scan a big amount of HD images that are all NON-FACES. All the boxes found will be False Negatives and those patches will be saved and added to the dataset as negatives and performed the **re-train** to enhance the model. The first step will add 15k negative patches, so the new dataset will be composed by 60k patches (64x64).

A source of HD images is [Landscape Pictures (Kaggle)](https://www.kaggle.com/datasets/arnaud58/landscape-pictures).

## Algorithms

- Viola-Jones algorithm [article](https://realpython.com/traditional-face-detection-python/#what-is-face-detection), [paper](https://www.cs.cmu.edu/~efros/courses/LBMV07/Papers/viola-cvpr-01.pdf), [example](https://scikit-image.org/docs/0.25.x/auto_examples/applications/plot_face_detection.html):
    * Selecting Haar-like features
    * Creating an integral image
    * Running AdaBoost training
    * Creating classifier cascades

- Histogram of Oriented Gradients HOG [paper](https://lear.inrialpes.fr/people/triggs/pubs/Dalal-cvpr05.pdf), [article](https://drone-vis.readthedocs.io/en/latest/face_detect/hog_face_detection.html), [example](https://scikit-image.org/docs/0.25.x/auto_examples/features_detection/plot_hog.html):
    * Preprocessing and normalization
    * Cell creation
    * Block normalization
    * Feature Vector Generation (high-dimentional vector)
    * Classification (Linar SVM)

- Convolutional Neural Networks (CNN): not considered due to the need of very large compute power in conflict with the limitations of the problem.

HOG is superior for the project because its block normalization makes it significantly more robust to the varied lighting conditions found in outdoor landscape photos. Unlike Viola-Jones, which relies on simple intensity differences, HOG captures complex structural gradients that allow the SVM to better distinguish between facial shapes and natural textures like rocks or trees. This flexibility is essential for the workflow, as it allows the model to fine-tune its decision boundary through the hard negative mining process you are currently performing.

# Dataset Building

## Download Datasets

In [ ]:
CURRENT_DIR = r"C:\Users\Tia\Desktop\ProfessionAI\Progetti\data-science-portfolio\[Data Science] face-detection"
model_filename = 'procam_face_classifier.pkl'

In [ ]:
# fairface_kaggle_path = "https://www.kaggle.com/datasets/aibloy/fairface/data"

### Fair Face -> positive

In [ ]:
def fairface_extract_balanced(zip_path, csv_inside_zip, target_dir, n_total=15000):
    '''
    Selectively extracts a balanced subset of images from a ZIP archive (FairFace) based on ethnic labels 
    found in an internal CSV file. The function shuffles the dataset to maintain diversity 
    in age and gender, then renames each file with its metadata during extraction.

    Args: 
        zip_path (str): Path to the source ZIP archive.
        csv_inside_zip (str): Relative path to the CSV file located inside the ZIP.
        target_dir (str): Destination directory where files will be extracted.
        n_total (int): Total number of samples to extract (default is 15000).

    Return: 
        None: The function extracts files directly to the disk and does not return an object.
    '''
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    print(f"Apertura archivio: {zip_path}")
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        # read csv inside zip folder
        with z.open(csv_inside_zip) as f:
            df = pd.read_csv(f)
        
        print("Dataset correctly read. Start balanced sampling...")

        # balanced sampling
        # shuffle enable random distribution
        df_shuffled = df.sample(frac=1, random_state=42)
        
        races = df_shuffled['race'].unique()
        n_per_race = n_total // len(races)
        
        selected_files_info = []
        
        for race in races:
            # take first n by race
            subset = df_shuffled[df_shuffled['race'] == race].head(n_per_race)
            for _, row in subset.iterrows():
                selected_files_info.append((row['file'], row['race'], row['age'], row['gender']))

        # dict to speed up the research in zip file
        files_to_find = {info[0]: info for info in selected_files_info}

        # selective extraction
        print(f"Extraction of {len(files_to_find)} selected samples...")
        
        # get the list of all path in the zip file
        zip_contents = z.namelist()
        count = 0
        
        for member in tqdm(zip_contents):

            clean_member_path = member.replace('FairFace/', '') 
            
            if clean_member_path in files_to_find:
                info = files_to_find[clean_member_path]
                race_label, age_label, gender_label = info[1], info[2], info[3]
                
                # final file name: ethnic_gender_age_originalname.jpg
                original_filename = os.path.basename(member)
                new_name = f"{race_label}_{gender_label}_{age_label}_{original_filename}"
                
                # extraction from the compressed folder
                source = z.open(member)
                with open(os.path.join(target_dir, new_name), "wb") as target:
                    target.write(source.read())
                
                count += 1
                
                if count >= n_total:
                    break

    print(f"\n--- COMPLETED ---")
    print(f"Images extracted in: {target_dir}")
    print(f"Total extracted files: {count}")

In [ ]:
ZIP_FAIRFACE_LOCAL_PATH = CURRENT_DIR + r"\downloads\archive.zip"
CSV_FAIRFACE_LOCAL_PATH = "FairFace/train_labels.csv" 
TARGET_FOLDER = CURRENT_DIR + r"\dataset_procam\positives\fairface_balanced"

#fairface_extract_balanced(ZIP_FAIRFACE_LOCAL_PATH, CSV_FAIRFACE_LOCAL_PATH, TARGET_FOLDER, n_total=15000)

### Intel Image dataset -> negative

In [ ]:
def setup_negatives_faces(total_patches=30000, images_to_use=5000, 
                          dir = "dataset_procam/negatives/patches_64x64",
                          crop=True, start=0):
    '''
    Downloads the Intel Natural Scenes dataset and extracts a large number of negative 
    (non-face) patches to build a balanced training set. The function samples images from 
    diverse landscape categories and crops random 64x64 patches from each to ensure 
    the model learns to distinguish faces from varied natural and urban backgrounds.

    Args: 
        total_patches (int): Total number of 64x64 patches to generate (default is 30000).
        images_to_use (int): Number of source images to sample from the dataset (default is 5000).
        crop (bool): If True, extracts random patches; if False download the full image (default is True).
        start (int): Index of the first image to consider in each category folder (default is 0).

    Return: 
        None: The function saves the generated patches directly to the local file system.
    '''
    # download from url
    print("Download Intel Natural Scenes From Kaggle...")
    kaggle_path = kagglehub.dataset_download("puneet6060/intel-image-classification")
    
    # dataset structure: seg_train/seg_train/[category]
    src_root = os.path.join(kaggle_path, "seg_train", "seg_train")
    categories = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
    
    patch_dir = dir
    os.makedirs(patch_dir, exist_ok=True)

    # proportion
    img_per_cat = images_to_use // len(categories)
    patches_per_img = total_patches // images_to_use
    
    print(f"Planning: {img_per_cat} images by category.")
    print(f"Extracion of {patches_per_img} patch from every image {150}x{150}.")

    patch_count = 0
    
    for cat in categories:
        cat_path = os.path.join(src_root, cat)
        all_imgs = sorted([f for f in os.listdir(cat_path) if f.lower().endswith(('.jpg', '.png'))])
        
        stop = start + img_per_cat
        selected_imgs = all_imgs[start:stop]
        
        for img_name in tqdm(selected_imgs, desc=f"Processing {cat}"):
            img = cv2.imread(os.path.join(cat_path, img_name))
            if img is None: continue
            
            if not crop:
                # if crop=False saving dimensions 150x150
                patch_name = f"full_{cat}_{patch_count}.jpg"
                cv2.imwrite(os.path.join(patch_dir, patch_name), img)
                patch_count += 1
                if patch_count >= total_patches: break
                continue
            
            h, w = img.shape[:2]
            
            # extract N random patches from the image
            for _ in range(patches_per_img):
                if patch_count >= total_patches: break
                
                y = random.randint(0, h - 64)
                x = random.randint(0, w - 64)
                
                patch = img[y:y+64, x:x+64]
                
                patch_name = f"neg_{cat}_{patch_count}.jpg"
                cv2.imwrite(os.path.join(patch_dir, patch_name), patch)
                patch_count += 1

    print(f"\n--- NEGATIVE DATASET COMPLETED ---")
    print(f"Total patches generated: {patch_count}")
    print(f"Folder: {os.path.abspath(patch_dir)}")

In [ ]:
#setup_negatives_faces(total_patches=30000, images_to_use=5000)

## Process images: resize positive

In [ ]:
def resize_positive_samples(src_dir, dst_dir, target_size=(64, 64)):
    '''
    Processes and resizes a collection of positive face samples (e.g., FairFace images) to a 
    uniform target size.

    Args: 
        src_dir (str): Path to the directory containing the original positive images.
        dst_dir (str): Path to the destination directory where resized images will be saved.
        target_size (tuple): The desired output dimensions (width, height), defaulting to 64x64.

    Return: 
        None: The function performs file I/O operations and saves the resized images directly to the disk.
    '''
    if not os.path.exists(dst_dir):
        os.makedirs(dst_dir)

    valid_extensions = ('.jpg', '.jpeg', '.png')
    images = [f for f in os.listdir(src_dir) if f.lower().endswith(valid_extensions)]
    
    print(f"Start resize of {len(images)} positive images...")

    count = 0
    for img_name in tqdm(images):
        src_path = os.path.join(src_dir, img_name)
        dst_path = os.path.join(dst_dir, img_name)
        
        img = cv2.imread(src_path)
        
        if img is not None:
            resized_img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
            cv2.imwrite(dst_path, resized_img)
            count += 1
        else:
            print(f"Error on loading for: {img_name}")

    print(f"\n--- COMPLETED ---")
    print(f"Images resized and saved in: {dst_dir}")
    print(f"Total images processed: {count}")

In [ ]:
DIR_POSITIVES_RAW = CURRENT_DIR + r"\dataset_procam\positives\fairface_balanced"
DIR_POSITIVES_READY = CURRENT_DIR + r"\dataset_procam\positives\fairface_64x64"

#resize_positive_samples(DIR_POSITIVES_RAW, DIR_POSITIVES_READY) review

## Mining dataset

In [ ]:
mining_dir = CURRENT_DIR + r"\dataset_procam\negatives\hard_mining"
# setup_negatives_faces(total_patches=100, images_to_use=100, dir=mining_dir, crop=False, start=850)

## Dataset index and split

In [ ]:
def create_procam_metadata(pos_dir, neg_dir):
    '''
    Create the dataset index with path and label columns, to be able to relate the picture (file path) and the label.
    
    Args:
        pos_dir (str): directory of positive samples
        neg_dir (str): directory of negative samples
        
    Return:
        data (pd.DataFrame): dict {path, label} in df format  
    '''
    data = []
    
    # scan positives
    pos_files = [os.path.join(pos_dir, f) for f in os.listdir(pos_dir) if f.lower().endswith(('.jpg', '.png'))]
    for f in pos_files:
        data.append({'path': f, 'label': 1})
        
    # scan negatives
    neg_files = [os.path.join(neg_dir, f) for f in os.listdir(neg_dir) if f.lower().endswith(('.jpg', '.png'))]
    for f in neg_files:
        data.append({'path': f, 'label': 0})

    return pd.DataFrame(data)

In [ ]:
def split_dataset(df, train_size=0.7, val_size=0.15, test_size=0.15):
    """
    Split a dataframe into train, validation and test
    
    Args:
        df (pd.DataFrame)
        train_size (float)
        val_size (float)
        test_size (float)
        
    Return:
        df_train, df_val, df_test (pd.DataFrame)
    """

    assert (train_size + val_size + test_size) == 1.0 # check sum = 1 
    
    # get Test set
    df_train_val, df_test = train_test_split(
        df, 
        test_size=test_size, 
        random_state=42, 
        stratify=df['label']
    )
    
    # get Train and Validaton set
    val_relative_size = val_size / (train_size + val_size)
    
    df_train, df_val = train_test_split(
        df_train_val, 
        test_size=val_relative_size, 
        random_state=42, 
        stratify=df_train_val['label']
    )
    
    return df_train, df_val, df_test

In [ ]:
DIR_POS = CURRENT_DIR + r"\dataset_procam\positives\fairface_64x64"
DIR_NEG = CURRENT_DIR + r"\dataset_procam\negatives\patches_64x64"

In [ ]:
df_metadata = create_procam_metadata(DIR_POS, DIR_NEG)
print(f"Total dataset: {len(df_metadata)} images.")

In [ ]:
df_train, df_val, df_test = split_dataset(df_metadata)

print(f"\nCheck distribution:")
print(f"TRAIN: {len(df_train)} images")
print(f"VAL:   {len(df_val)} images")
print(f"TEST:  {len(df_test)} images")

# Verify stratification
print("\nPercentage faces in Train:", (df_train['label'] == 1).mean())
print("Percentage faces in Val: ", (df_val['label'] == 1).mean())
print("Percentage faces in Test: ", (df_test['label'] == 1).mean())

# Model: face classifier

- Feature Extraction: HOG Trasformer
- Preprocessing: normalization
- Classification model LinearSVC
- Hyperparameter tuning

In [ ]:
class HOGTransformer(BaseEstimator, TransformerMixin):
    '''
    Custom Scikit-Learn transformer that extracts Histogram of Oriented Gradients (HOG) features 
    from a collection of images. 
    
    It automatically handles color-to-grayscale conversion 
    and flattens each image into a high-dimensional feature vector suitable for machine 
    learning classifiers like SVM.

    Args: 
        orientations (int): Number of orientation bins (default is 9).
        pixels_per_cell (tuple): Size (in pixels) of a cell (default is 8x8).
        cells_per_block (tuple): Number of cells in each block (default is 2x2).
        block_norm (str): Block normalization method (default is 'L2-Hys').

    Return: 
        HOG_features (np.array): A 2D array of shape (n_samples, n_features) containing the extracted descriptors.
    '''
    def __init__(self, orientations=9, pixels_per_cell=(8, 8), 
                 cells_per_block=(2, 2), block_norm='L2-Hys'):
        self.orientations = orientations
        self.pixels_per_cell = pixels_per_cell
        self.cells_per_block = cells_per_block
        self.block_norm = block_norm

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # X list of images (numpy arrays)
        def get_hog(img):
            # turn to grey scale if it's a color image
            if len(img.shape) > 2:
                img = rgb2gray(img)
            return hog(img, 
                       orientations=self.orientations,
                       pixels_per_cell=self.pixels_per_cell,
                       cells_per_block=self.cells_per_block,
                       block_norm=self.block_norm)
        
        return np.array([get_hog(img) for img in X])

In [ ]:
def load_images_from_df(df):
    images = []
    for path in df['path']:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        images.append(img)
    return np.array(images), df['label'].values

In [ ]:
X_train_raw, y_train = load_images_from_df(df_train)
X_val_raw, y_val = load_images_from_df(df_val)
X_test_raw, y_test = load_images_from_df(df_test)

In [ ]:
# Pipeline definition
procam_model = Pipeline([
    ('hog', HOGTransformer()),      
    ('scaler', StandardScaler()),   
    ('svc', LinearSVC(dual=False, max_iter=2000)) 
])

# Tuning with GridSearch
param_grid = {
    'svc__C': [0.01, 0.1, 1, 10]
}

In [ ]:
grid = GridSearchCV(procam_model, param_grid, cv=3, n_jobs=-1, verbose=2)
grid.fit(X_train_raw, y_train)

In [ ]:
def plot_learning_curve(estimator, X, y):
    '''
    Plot the learning curve of of a given estimator by plotting training and 
    cross-validation scores against various training set sizes.
    
    Args: 
        estimator (object): The machine learning model instance (e.g., SVM) to be evaluated.
        X (np.array): Feature matrix of shape (n_samples, n_features).
        y (np.array): Target vector of shape (n_samples,).

    Return: 
        None: The function generates and displays a Matplotlib plot and does not return any value.
    '''
    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=3, n_jobs=-1, 
        train_sizes=np.linspace(0.1, 1.0, 5), verbose=1
    )

    train_mean = np.mean(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)

    plt.figure(figsize=(8, 6))
    plt.plot(train_sizes, train_mean, 'o-', label="Training score")
    plt.plot(train_sizes, test_mean, 'o-', label="Cross-validation score")
    plt.title("Learning Curve (HOG + SVM)")
    plt.xlabel("N° training samples")
    plt.ylabel("Accuracy")
    plt.legend(loc="best")
    plt.grid()
    plt.show()

In [ ]:
# plot C param vs performance
results = pd.DataFrame(grid.cv_results_)
plt.plot(results['param_svc__C'], results['mean_test_score'], marker='o')
plt.xscale('log')
plt.xlabel('Param C (Regularization)')
plt.ylabel('Mean CV Accuracy')
plt.title('C effect on performance')
plt.show()

In [ ]:
plot_learning_curve(grid.best_estimator_, X_train_raw, y_train)

In [ ]:
best_index = grid.best_index_
train_score = grid.cv_results_['mean_test_score'][best_index]
val_score = grid.score(X_val_raw, y_val)                     

print(f"Accuracy Train (CV): {train_score:.4f}")
print(f"Accuracy Validation: {val_score:.4f}")
print(f"Gap: {train_score - val_score:.4f}")

In [ ]:
def evaluate_procam_model(model, X_test, y_test):
    '''
    Generates a comprehensive performance report for an input model, including a textual 
    classification report and graphical visualizations for the Confusion Matrix, ROC Curve, 
    and Precision-Recall Curve. 
    
    It automatically handles both probability-based and 
    decision-function-based scores for compatibility with different estimators.

    Args: 
        model (object): The trained classifier to evaluate (supports predict_proba or decision_function).
        X_test (np.array/pd.DataFrame): Testing feature matrix.
        y_test (np.array/pd.Series): True ground truth labels.

    Return: 
        None: The function prints the report and displays Matplotlib plots directly.
    '''
    y_pred = model.predict(X_test)
    # some SVM model use decision_function
    if hasattr(model, "predict_proba"):
        y_scores = model.predict_proba(X_test)[:, 1]
    else:
        y_scores = model.decision_function(X_test)

    # report
    print("="*30)
    print("PERFORMANCE REPORT")
    print("="*30)
    print(classification_report(y_test, y_pred, target_names=['No Face (0)', 'Face (1)']))

    # plot
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0],
                xticklabels=['No Face', 'Face'], yticklabels=['No Face', 'Face'])
    ax[0].set_title('Confusion Matrix')
    ax[0].set_ylabel('True')
    ax[0].set_xlabel('Predicted')

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_scores)
    roc_auc = auc(fpr, tpr)
    ax[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    ax[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax[1].set_title('CurvaeROC')
    ax[1].set_xlabel('False Positive Rate')
    ax[1].set_ylabel('True Positive Rate')
    ax[1].legend(loc="lower right")

    # Precision-Recall curve
    precision, recall, _ = precision_recall_curve(y_test, y_scores)
    ax[2].plot(recall, precision, color='green', lw=2)
    ax[2].set_title('Precision-Recall curve')
    ax[2].set_xlabel('Recall')
    ax[2].set_ylabel('Precision')

    plt.tight_layout()
    plt.show()

In [ ]:
# on validation set
evaluate_procam_model(grid.best_estimator_, X_val_raw, y_val)

In [ ]:
# on test set
evaluate_procam_model(grid.best_estimator_, X_test_raw, y_test)

💡 **Observation**

The performace showed by the model are excellent, even on the validation and test set.

- Overfitting gap irrelevant between Validation and Train Accuracy, also the learning curve is stable close to 1.
- Precision: means model prredict a face, get right **97%** of the times, false negatives only **63** on 4500 negatives.
- Recall: model recognizes **98%** of the faces in the set, only false positive **45** on 2249 total faces.
- F1-Score: very high confirming the power of the model.
- ROC Curve: almost 1, indicates perfect separation between classes.
- C selection: the diagram shows that for value **($10^{-2}$)** mean cross validatoin is the highest. This is coherent in the controlling of overfitting: low C need a stronger regularization.

### Save pipeline

In [ ]:
joblib.dump(grid.best_estimator_, model_filename)

# Detector

- Preprocess image input
- Pyramid scaling
- Sliding window 
- HOG Features extraction
- Prediction on SVM
- Obtain coordinates list from positives
- Non-Maximum Suppression
- Return list of final coordinates

## Sliding window

In [ ]:
def sliding_window(image, step_size=8, window_size=(64, 64)):
    '''
    Implements a sliding window mechanism that traverses an image using a fixed-size window and a defined step size.

    Args: 
        image (np.array): The input image (usually grayscale) to be scanned.
        step_size (int): The number of pixels to skip in both horizontal and vertical directions (default is 8).
        window_size (tuple): The (height, width) dimensions of the scanning window (default is 64x64).

    Return: 
        generator: A Python generator yielding tuples of (x, y, window_patch) for each position.
    '''
    for y in range(0, image.shape[0] - window_size[0], step_size):
        for x in range(0, image.shape[1] - window_size[1], step_size):
            yield (x, y, image[y:y + window_size[0], x:x + window_size[1]])

In [ ]:
def test_sliding_window(image_path, step_size=32):
    '''
    Test for the sliding window function implemented
    '''
    img = cv2.imread(image_path)
    temp_img = img.copy()
    
    counter = 0
    # show only 1 on 10 image
    for (x, y, window) in sliding_window(img, step_size=step_size):
        if counter % 10 == 0:
            cv2.rectangle(temp_img, (x, y), (x + 64, y + 64), (0, 255, 0), 2)
        counter += 1
    
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(temp_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Test Sliding Window (showed 1 every 10). Total windows: {counter}")
    plt.show()

In [ ]:
test_image = CURRENT_DIR + r"\dataset_procam\negatives\10082.jpg"
image_2 = CURRENT_DIR + r"\downloads\landscape_hd_1.jpg"
test_sliding_window(image_2, step_size=100)

## Image Pyramid

In [ ]:
def image_pyramid(image, scale=1.5, min_size=(64, 64)):
    '''
    Generates a multi-scale representation of an input image by iteratively downsizing it
    using a constant scale factor (until the minimum size).

    Args:
        image (np.array): The original input image to be scaled.
        scale (float): The factor by which the image dimensions are reduced at each step (default is 1.5).
        min_size (tuple): A (width, height) threshold that stops the pyramid generation (default is 64x64).

    Return:
        generator: a python generator that yields the image at each level of the pyramid, starting from the original.
    '''
    yield image

    current_image = image
    while True:
        # calculate dimensions on the previous level shape 
        w = int(current_image.shape[1] / scale)
        h = int(current_image.shape[0] / scale)
        
        # stop if scale is too little
        if w < min_size[0] or h < min_size[1]:
            break
            
        # resize
        current_image = cv2.resize(current_image, (w, h), interpolation=cv2.INTER_AREA)
        
        yield current_image

In [ ]:
def test_pyramid(image_path):
    '''
    Test for the implementation of the pyramid function
    '''
    img = cv2.imread(image_path)
    if img is None:
        print("Error: Image not found.")
        return
        
    # force to RGB conversion
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # take just the first 3 channels
    if img.shape[2] == 4:
        img = img[:, :, :3]

    print(f"Original dimension: {img.shape}")
    
    for i, layer in enumerate(image_pyramid(img, scale=1.5)):
        print(f"Level {i}: {layer.shape}")
        
        plt.figure(figsize=(2, 2))
        plt.imshow(layer)
        plt.title(f"L {i}: {layer.shape[1]}x{layer.shape[0]}")
        plt.axis('off')
        plt.show()

In [ ]:
test_image = CURRENT_DIR + r"\dataset_procam\negatives\10082.jpg"
test_pyramid(test_image)

## Non-Maximum Suppression (NMS)

In [ ]:
def non_max_suppression(boxes, probs, overlap_thresh=0.3):
    '''
    Implements the Non-Maximum Suppression (NMS) algorithm to filter overlapping 
    bounding boxes. It identifies the box with the highest confidence score and 
    removes all other candidate boxes that have an Intersection over Union (IoU) 
    overlap greater than the specified threshold, ensuring only the most prominent 
    detections remain.

    Args:
        boxes (list/np.array): A collection of bounding boxes, each defined as [x1, y1, x2, y2].
        probs (list/np.array): Confidence scores or probability values associated with each box.
        overlap_thresh (float): The IoU threshold for suppressing overlapping boxes (default is 0.3).

    Returns:
        np.array: An array of selected bounding boxes, cast as integers, after suppression.
    '''
    if len(boxes) == 0:
        return []

    boxes = np.array(boxes)
    probs = np.array(probs)

    # coordinates
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    areas = (x2 - x1 + 1) * (y2 - y1 + 1)  # area of the boxes
    
    idxs = np.argsort(probs) # sort by prob

    pick = []
    while len(idxs) > 0:
        last = len(idxs) - 1
        i = idxs[last]
        pick.append(i)

        # find min, max coordinates
        xx1 = np.maximum(x1[i], x1[idxs[:last]])
        yy1 = np.maximum(y1[i], y1[idxs[:last]])
        xx2 = np.minimum(x2[i], x2[idxs[:last]])
        yy2 = np.minimum(y2[i], y2[idxs[:last]])

        w = np.maximum(0, xx2 - xx1 + 1)
        h = np.maximum(0, yy2 - yy1 + 1)

        # calculate overlap ratio
        overlap = (w * h) / areas[idxs[:last]]

        # delete indexes with excessive ovelap
        idxs = np.delete(idxs, np.concatenate(([last], np.where(overlap > overlap_thresh)[0])))

    return boxes[pick].astype("int")

In [ ]:
def test_nms_visual(image_path):
    '''
    Test on the implementation of the function of mns
    '''
    orig_img = cv2.imread(image_path)
    if orig_img is None:
        print(f"Error: Image not found {image_path}")
        return
    
    img_before = orig_img.copy()
    img_after = orig_img.copy()
    
    # simulation of bounding boxes and scores
    simulated_boxes = [
        [200, 150, 264, 214], 
        [205, 153, 269, 217], 
        [198, 155, 262, 219], 
        [220, 170, 284, 234], 
        [50, 50, 114, 114]    
    ]
    simulated_scores = [0.92, 0.98, 0.85, 0.70, 0.95]
    
    print(f"Total simulated rectangular: {len(simulated_boxes)}")
    
    # image before nms
    for i, (startX, startY, endX, endY) in enumerate(simulated_boxes):
        score = simulated_scores[i]
        # red rectangle
        cv2.rectangle(img_before, (startX, startY), (endX, endY), (0, 0, 255), 2)
        cv2.putText(img_before, f"{score:.2f}", (startX, startY - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        
    # application nms
    overlap_threshold = 0.3
    final_boxes = non_max_suppression(np.array(simulated_boxes), np.array(simulated_scores), overlap_thresh=overlap_threshold)
    
    print(f"Rectangle after NMS: {len(final_boxes)}")
    
    # image after nms
    for (startX, startY, endX, endY) in final_boxes:
        # green rectangle
        cv2.rectangle(img_after, (startX, startY), (endX, endY), (0, 255, 0), 3)
        
    # plot
    plt.figure(figsize=(14, 7))

    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img_before, cv2.COLOR_BGR2RGB))
    plt.title(f"PRIMA dell'NMS ({len(simulated_boxes)} box sovrapposti)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(cv2.cvtColor(img_after, cv2.COLOR_BGR2RGB))
    plt.title(f"DOPO l'NMS ({len(final_boxes)} box puliti)")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()


In [ ]:
IMAGE_PATH = CURRENT_DIR+ r"\dataset_procam\positives\faccia_mattia_piccola.jpg" 

test_nms_visual(IMAGE_PATH)

## Crop image from box coordinates

In [ ]:
def get_crop_from_box(image_path, box, target_size=(64, 64)):
    '''
    Extracts, clips, and resizes a specific sub-region (patch) from an image based 
    on provided bounding box coordinates. The function ensures the crop stays within 
    the image boundaries to prevent errors and applies area-based interpolation 
    during resizing to maintain feature quality for HOG extraction.

    Args: 
        image_path (str): File system path to the source image.
        box (list or tuple): Bounding box coordinates in the format [x1, y1, x2, y2].
        target_size (tuple): The desired output dimensions (width, height), defaulting to 64x64.

    Return: 
        np.array or None: The processed image patch as a NumPy array, or None if the 
                            image cannot be loaded or the resulting crop is empty.
    '''
    img = cv2.imread(image_path)
    if img is None:
        return None

    x1, y1, x2, y2 = box

    # border protection
    h, w = img.shape[:2]
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(w, x2)
    y2 = min(h, y2)

    crop = img[y1:y2, x1:x2]

    if crop.size == 0:
        return None

    crop_resized = cv2.resize(crop, target_size, interpolation=cv2.INTER_AREA)

    return crop_resized

## Full detector

In [ ]:
def procam_detector(image_path, model, min_probability=0.8, step_size=12, scale_factor = 1.2, nms=True):
    '''
    Performs face detection on an input image using a sliding window approach across an 
    image pyramid. It extracts HOG features for each window, evaluates them with the 
    provided model, and filters detections using Non-Maximum Suppression (NMS) to 
    remove overlapping boxes.

    Args: 
        image_path (str): File system path to the image to be analyzed.
        model (object): Trained classifier (or pipeline) used to score image windows.
        min_probability (float): Threshold for the decision function to consider a detection valid (default is 0.8).
        step_size (int): The number of pixels to skip in each sliding window step; lower values increase precision but reduce speed (default is 12).
       scale_factor (float): The multiplier used to downsample the image at each pyramid level (default is 1.2).

    Return: 
        orig (np.array): A copy of the original image in BGR format with green bounding boxes drawn around detected faces.
    '''
    image = cv2.imread(image_path)
    orig = image.copy()
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) # HOG needs gray

    detections = []
    probs = []
    
    # scan params
    (winW, winH) = (64, 64)

    print("Scanning...")
    current_scale = 1.0
    
    for layer in image_pyramid(image, scale=scale_factor):
        for (x, y, window) in sliding_window(layer, step_size=step_size):
            
            # prediction
            score = model.decision_function([window])[0]
            if score > min_probability:
                # original scale coordinates
                real_x = int(x * current_scale)
                real_y = int(y * current_scale)
                real_w = int(winW * current_scale)
                real_h = int(winH * current_scale)
                
                detections.append((real_x, real_y, real_x + real_w, real_y + real_h))
                probs.append(score)
        
        current_scale *= scale_factor

    # apply NMS to clean results
    if nms:
        final_boxes = non_max_suppression(detections, probs, overlap_thresh=0.15)
    else:
        final_boxes = detections
    
    # show results
    for (startX, startY, endX, endY) in final_boxes:
        cv2.rectangle(orig, (startX, startY), (endX, endY), (0, 255, 0), 2)
    
    return orig, final_boxes

In [ ]:
# load frozen model
model = joblib.load(model_filename)

In [ ]:
def test_detector(model, image_path, step_size=24, min_probability=0.5, show_boxes=True, nms=True):
    result_img, boxes = procam_detector(image_path, model, step_size=step_size, min_probability=min_probability, nms=nms)
    if show_boxes:
        risultato_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(risultato_rgb)
        plt.axis('off') 
        plt.title("Face Detection result")
        plt.show()
    
    return boxes

In [ ]:
IMAGE_PATH = CURRENT_DIR + r"\dataset_procam\positives\indonesia2.jpg"
boxes = test_detector(model, IMAGE_PATH)

In [ ]:
# crop test
crop_img = get_crop_from_box(image_path=IMAGE_PATH, box=boxes[5])
risultato_rgb = cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 8))
plt.imshow(risultato_rgb)
plt.axis('off') 
plt.title("Crop result")
plt.show()

In [ ]:
test_image2 = CURRENT_DIR+ r"\dataset_procam\positives\indonesia.jpg" 
boxes = test_detector(model, test_image2)

In [ ]:
crop_img = get_crop_from_box(image_path=test_image2, box=boxes[10])
risultato_rgb = cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 8))
plt.imshow(risultato_rgb)
plt.axis('off') 
plt.title("Crop result")
plt.show()

In [ ]:
len(boxes)

# Hard Negative Mining

In [ ]:
image_neg = CURRENT_DIR + r"\downloads\landscape_hd_1.jpg"
boxes = test_detector(model, image_path=image_neg)

In [ ]:
len(boxes)

## Scanning negatives

In [ ]:
dir_mining_patch = CURRENT_DIR + r"\dataset_procam\negatives\hard_mining\patches_64x64"

In [ ]:
def get_patches_for_mining(path, file_list, destination_dir, lower_limit=None, upper_limit=None): 
    '''
    Performs Hard Negative Mining by scanning a set of images to identify and extract 
    false positive detections. The function runs the current detector on images known 
    not to contain faces; any detected boxes are considered "hard negatives." These 
    patches are cropped, resized to 64x64, and saved to a destination folder to be 
    used for model retraining.

    Args:
        path (str): The base directory path containing the source images.
        file_list (list): A list of filenames (strings) to be scanned within the path.
        destination_dir (str): The directory where the extracted hard negative patches will be saved.

    Return:
        None: The function saves the extracted patches directly to the disk and prints a summary of the results.
    '''
    hard_negs_dict = {}

    print(f"Phase 1: Scan of {len(file_list)} images...")
    
    if upper_limit is not None:
        file_list = file_list[:upper_limit]
    if lower_limit is not None:
        file_list = file_list[lower_limit:]
        
    print(f"List length= {len(file_list)}")

    for fname in tqdm(file_list):
        img_path = os.path.join(path, fname)
        # run the test detector
        boxes = test_detector(model, img_path, nms=False)
        
        if len(boxes) > 0:
            hard_negs_dict[img_path] = boxes

    total_found = sum(len(b) for b in hard_negs_dict.values())
    print(f"\nScan completed. Found {total_found} ptential Hard Negatives on {len(hard_negs_dict)} images.")
    
    print(f"Phase 2: Extraction and save of patches in {destination_dir}...")

    patch_counter = 0
    for img_path, boxes in tqdm(hard_negs_dict.items()):
        img = cv2.imread(img_path)
        if img is None: continue
        h, w = img.shape[:2]
        
        base_name = os.path.basename(img_path).split('.')[0]
        
        for i, (x1, y1, x2, y2) in enumerate(boxes):
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
        
            
            crop = img[y1:y2, x1:x2]
            if crop.size == 0: continue
            
            crop_resized = cv2.resize(crop, (64, 64), interpolation=cv2.INTER_AREA)
            
            patch_filename = f"hm_{base_name}_{i}.jpg"
            cv2.imwrite(os.path.join(destination_dir, patch_filename), crop_resized)
            patch_counter += 1

    print(f"\Done! Saved {patch_counter} patches.")

In [ ]:
# scan of 96 intel dataset images
path = CURRENT_DIR + r"\dataset_procam\negatives\hard_mining"
mining_file_list_intel = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
#get_patches_for_mining(path, mining_file_list_intel, dir_mining_patch)

In [ ]:
# scan of my downloads 
path = CURRENT_DIR + r"\downloads\mining_1"
mining_file_list_hd = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
#get_patches_for_mining(path, mining_file_list_hd, dir_mining_patch)

In [ ]:
# scan of first 100 in Landscape Pictures Kaggle dataset
path = CURRENT_DIR + r"\downloads\archive_1"
mining_file_list_hd = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
#get_patches_for_mining(path, mining_file_list_hd, dir_mining_patch, upper_limit=100)

In [ ]:
# scan 101-150 in Landscape Pictures Kaggle dataset
path = CURRENT_DIR + r"\downloads\archive_1"
mining_file_list_hd = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
#get_patches_for_mining(path, mining_file_list_hd, dir_mining_patch, upper_limit=150, lower_limit=101)

## Re-train, optuna

In [ ]:
def create_neg_metadata(neg_dir, limit=15000):
    '''
    Generates a DataFrame containing file paths and labels (0) for negative samples.
    
    Args:
        neg_dir (str): directory of negative samples
        limit (int): Maximum number of images to include in the metadata
        
    Return:
        data (pd.DataFrame): dict {path, label} in df format  
    '''
    data = []
    files = sorted([f for f in os.listdir(neg_dir) if f.lower().endswith(('.jpg', '.png'))])
    selected_files = files[:limit]
    
    for f in selected_files:
        data.append({
            'path': os.path.join(neg_dir, f), 
            'label': 0
        })

    return pd.DataFrame(data)

In [ ]:
DIR_NEG_MINING = CURRENT_DIR + r"\dataset_procam\negatives\hard_mining\patches_64x64"

df_metadata_hm = create_neg_metadata(DIR_NEG_MINING, limit=15000)
df_metadata_hm = df_metadata_hm[df_metadata_hm['label']==0]
print(f"Total dataset: {len(df_metadata_hm)} images.")

In [ ]:
df_metadata_mining = pd.concat([df_metadata, df_metadata_hm], ignore_index=True)

print(f"Verification of length:")
print(f"Original dataset: {len(df_metadata)} || Mining dataset: {len(df_metadata_mining)}")
print(f"Count positives = {len(df_metadata_mining[df_metadata_mining['label']==1])} || Count negatives = {len(df_metadata_mining[df_metadata_mining['label']==0])}")

In [ ]:
df_train, df_val, df_test = split_dataset(df_metadata_mining)
X_train_raw, y_train = load_images_from_df(df_train)
X_val_raw, y_val = load_images_from_df(df_val)
X_test_raw, y_test = load_images_from_df(df_test)

In [ ]:
def objective(trial):
    # research space
    c_param = trial.suggest_float('svc__C', 1e-4, 1e2, log=True)
    
    # update param in pipeline
    procam_model.set_params(svc__C=c_param)
    
    # calcolate cross validation
    score = cross_val_score(procam_model, X_train_raw, y_train, n_jobs=-1, cv=5)
    
    return score.mean()

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print(f"Best param C: {study.best_params['svc__C']}")
print(f"Best Accuracy: {study.best_value}")

In [ ]:
best_trial = study.best_trial

# apply best params to pipeline
procam_model.set_params(**best_trial.params)
procam_model.fit(X_train_raw, y_train)
train_score = study.best_value 
val_score = procam_model.score(X_val_raw, y_val)

print(f"Accuracy Train (Best Trial): {train_score:.4f}")
print(f"Accuracy Validation: {val_score:.4f}")
print(f"Gap: {abs(train_score - val_score):.4f}")

evaluate_procam_model(procam_model, X_val_raw, y_val)
evaluate_procam_model(procam_model, X_test_raw, y_test)

### Save re-trained model

In [ ]:
model_filename_2 = model_filename + "_2"
joblib.dump(procam_model, model_filename_2)

## Test new detector

In [ ]:
model_2 = joblib.load(model_filename_2)

In [ ]:
IMAGE_PATH = CURRENT_DIR + r"\dataset_procam\positives\indonesia2.jpg"
boxes = test_detector(model_2, IMAGE_PATH, min_probability=0.8)

In [ ]:
test_image2 = CURRENT_DIR+ r"\dataset_procam\positives\indonesia.jpg" 
boxes = test_detector(model_2, test_image2, min_probability=0.8)

# Test Optuna with PCA

In [ ]:
# pca pipeline
procam_model_pca = Pipeline([
    ('hog', HOGTransformer()),      
    ('scaler', StandardScaler()),   
    ('pca', PCA(svd_solver='randomized')),
    ('svc', LinearSVC(dual=False, max_iter=2000)) 
])

# Optuna object function updated
def objective(trial):

    n_components = trial.suggest_int("pca__n_components", 100, 1000)
    c_param = trial.suggest_float("svc__C", 1e-5, 1e2, log=True)
    
    params = {
        'pca__n_components': n_components,
        'svc__C': c_param
    }
    procam_model_pca.set_params(**params)
    
    procam_model_pca.fit(X_train_raw, y_train)
    return procam_model.score(X_val_raw, y_val)

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print(f"Best param C: {study.best_params['svc__C']}")
print(f"Best param PCA components: {study.best_params['pca__n_components']}")
print(f"Best Accuracy: {study.best_value}")

In [ ]:
best_trial = study.best_trial
procam_model_pca.set_params(**best_trial.params)
procam_model_pca.fit(X_train_raw, y_train)
train_score = study.best_value 
val_score = procam_model_pca.score(X_val_raw, y_val)

print(f"Accuracy Train (Best Trial): {train_score:.4f}")
print(f"Accuracy Validation: {val_score:.4f}")
print(f"Gap: {abs(train_score - val_score):.4f}")

evaluate_procam_model(procam_model_pca, X_val_raw, y_val)
evaluate_procam_model(procam_model_pca, X_test_raw, y_test)

In [ ]:
model_filename_pca = model_filename + "_pca"
joblib.dump(procam_model, model_filename_pca)

## Test full detector with model-pca

In [ ]:
model_pca = joblib.load(model_filename_pca)

In [ ]:
IMAGE_PATH = CURRENT_DIR + r"\dataset_procam\positives\indonesia2.jpg"
boxes = test_detector(model_pca, IMAGE_PATH, min_probability=0.8)

In [ ]:
test_image2 = CURRENT_DIR+ r"\dataset_procam\positives\indonesia.jpg" 
boxes = test_detector(model_pca, test_image2, min_probability=0.8)

# Conclusions

On first hand the classification model performed very well each on validation and test set, but put in the detector produced too many false positives.

The Hard Negative Mining joined with an Optuna fine tuning showed a big increase in the detector response.

The comparison between retrained models with or without PCA showed the same metrics but the PCA pipeline produced more False Negatives both in validation and test, so we'd like to use **procam_face_classifier.pkl_2 ** model in the detector.

The detector showed on the test still a lot of false negatives and some further experiments can consider:
- HOG parameters
    * orientations: The number of bins in the gradient (usually between 6 and 12). More orientations capture finer details but increase the size of the vector.
    * pixels_per_cell: The cell size (8x8), (16x16). Determines the scale of the details captured.
    * cells_per_block: How many cells to normalize together (e.g., 2x2 or 3x3). Essential for robustness to light variations.
    * block_norm: The normalization method (L1, L1-sqrt, or L2-Hys). L2-Hys is often the most robust for faces.
- Pipeline preprocessing
    * power trnsformer: apply a transformation (such as Yeo-Johnson) to make the data more “Gaussian” before SVM.
- SVM parameters
    * loss:  try between *hinge* and *squared_hinge*.
    * class_weight: Since you have more negatives (60k) than positives (15k), you could set class_weight=‘balanced’ to give more weight to faces when calculating the error.
- Detector parameters (inference)
    * scale (Image Pyramid): The reduction factor between one level and another in the pyramid (e.g., $1.2$ vs. $1.5$). The smaller it is, the more likely it is to find distant faces, but the longer the calculation time.
    * step_size: The sliding window jump. A small step (e.g., $4$ or $8$) is more accurate but much slower.
    * min_probability (Decision Threshold): The SVM confidence threshold above which a box is considered “turned.”
    * nms_threshold: The overlap value (IoU) for Non-Maximum Suppression. A low value eliminates many nearby boxes, while a high value keeps more of them.

- Dataset augmetation
    * positives: add samples to introduce variance from other datasets (like those cited in the intro), better if the faces are profiles or occulted with glasses, hat, scarf,... This will allow the model to learn more patterns.
    * negatives: perform recursively the Hard Negative Mining strategy and add the found false positives to the negative dataset.